### This ARTICLES notebook 

- Loads journal data into the database  
- Finds the ISSN of Domingo's Incites journals  
- Examines the "completeness" of the Incites journals

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publicaiotn date and type (articles, etc)


In [125]:
%run common_setup.ipynb

In [126]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_journals(self):
        # Extract inCites-OpenAlex journal table (ISSN and OA journal_id)
        try:
            self.journals = self.db.sql("SELECT * FROM project.sources_oa_incites").df().\
                rename(columns={'id': 'source_id'}).sort_values('source_id')[:].reset_index(drop=True)
            print(f'{self.journals.shape = }\n{self.journals.head()}')
        except Exception as e:
            print('need to load sources_oa_incites from CSV')
            self.db.sql("CREATE OR REPLACE TABLE project.sources_oa_incites AS (SELECT * FROM read_csv('../DATA/sources_oa_incites.csv'))")
            self.journals = self.db.sql("SELECT * FROM project.sources_oa_incites").df().\
                rename(columns={'id': 'source_id'}).sort_values('source_id')[:].reset_index(drop=True)
            print(f'{self.journals.shape = }\n{self.journals.head()}')
        return

    
    def extract_works_by_journal(self):
        # Extract OA works for the journal set, for publication years 2010+ to now
        hold = []
        for row in self.journals.itertuples():
            # if row.Index != 139:
            #     continue
            source_id = row.source_id
            reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
            if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
                oa = self._filter_works(works=oa)
                print(f'{row.Index = } {oa.shape = }')
                self._sql_appender(df=oa, row=row.Index)
            else:
                print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
            if row.Index % 25 == 0:
                print(f'{row.Index}/{len(self.journals)} completed')             
        return
    
    def _filter_works(self, works=None):
        keep_columns = ["id", "doi", "title", "publication_year", "primary_location", "type",
                "countries_distinct_count", "institutions_distinct_count", "fwci", "has_fulltext", 
                "cited_by_count", "biblio", "is_retracted", "is_paratext", 
                "referenced_works_count", "cited_by_api_url", "updated_date", "created_date", 
                "authorships", "referenced_works", "topics"]
        cols = [c for c in works.columns if c in keep_columns] + [c for c in works.columns if 'biblio.' in c or 'primary_location.source' in c or 'primary_topic.' in c]
        works = works[cols].fillna(' ')
        return works[works.id != 'https://openalex.org/works/W4285719527'] # THIS IS A DISCARDED WORK WITH ENORMOUS CITATIONS
    
    def _sql_appender(self, df=None, row=None):
        self._drop_older_duplicated_rows(df=df)
        try:
            if row == 0:
                sql = "CREATE OR REPLACE TABLE project.raw AS (SELECT * FROM df)"
            else:
                sql = "INSERT INTO project.raw BY NAME (SELECT * FROM df)"
            self.db.sql(sql)
        except Exception as e:
            print(f'CONVERTING df to SQL table project.raw {row = } {e = }')           
            print(f'{df.shape = }\n{df.columns = }\n{df.head()}')
            # df = df.head(1)
            # print(f'{df.shape = }\n{df.columns = }\n{df.head()}')
            # self.db.sql("CREATE OR REPLACE TABLE project.junk AS (SELECT * FROM df)")
            # self.db.sql("SELECT * FROM project.junk").show()
        # self.db.sql("SELECT * FROM project.raw").show()
        # print(f'{df.shape = }\n{df.columns = }\n{df.head()}')
        return
    
    def _drop_older_duplicated_rows(self, df=None):
        rows = df.shape[0]
        df = df.sort_values("updated_date", ascending=False).drop_duplicates(subset='id')
        if rows != df.shape[0]:
            print(f'DROP OLDER ROWS {df.shape = }')
        return
 

In [127]:
class ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def build_author_list(self):
        df = self.db.sql("SELECT DISTINCT author_id FROM project.authorships WHERE author_id NOT NULL ORDER BY author_id").df()
        self.author_ids = df.author_id.str.replace('https://openalex.org/', '').tolist()
        print(f'{len(self.author_ids) = } {self.author_ids[:4] = }')
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        hold = []
        block_length = 100
        start = 0
        block_total = len(self.author_ids)//block_length + 1
        print(f'extract authors {start = } {block_length = } {block_total = }')
        for block_count in range(block_total):
            authors = '|'.join(self.author_ids[start: start+block_length])
            start = start + block_length
            reader = rf'Authors().filter(id="{authors}")'
            if oa := self._read_openalex(reader=reader):
                # print(f'EXTRACTED {len(oa) = } authors FOR {authors = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have authors for {authors = }')
            if block_count % 1 == 0:
                print(f'{block_count = } {block_count*block_length}/{len(self.author_ids)} completed')
        self._load_authors(hold=hold)
        return
    
    def _read_openalex(self, reader=None):
        # using the pyalex API call "reader" as key, run the API call, cache the response, and return it (JSON) 
        if result := self.cache.get(reader):
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        self.cache[reader] = result
        return result
    
    def _load_authors(self, hold=None):
        df = pd.DataFrame(hold).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        for col in ['2yr_mean_citedness', 'h_index', 'i10_index']:
            df[col] = [s.get(col) for s in df.summary_stats]
        cols = ['author_id', 'orcid', 'author_name', 'display_name_alternatives', 'works_count', 'cited_by_count', '2yr_mean_citedness', 'h_index', 'i10_index']
        df = df[cols]
        df[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in df.author_name]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.authors_full AS SELECT * FROM df")
        self.db.sql("SELECT * FROM project.authors_full").show()
        self.db.sql("SELECT count(*) FROM project.authors_full").show()
        return        

In [128]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        self.db.sql("CREATE OR REPLACE TABLE econ.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        return

    
    def match_sample(self):
        print('_match_sample')
        sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        authors = self.db.sql("SELECT * FROM econ.authors").df().sort_values(['cited_by_count', 'works_count'], ascending=[False, False])\
            .reset_index()[['author_id', 'author_name', 'orcid', 'display_name_alternatives', 'cited_by_count', 'works_count']]
        # authors = authors[authors.cited_by_count>100].dropna()
        for row in authors.itertuples():
            authors.at[row.Index, 'display_name_alternatives'] = [normalise_name(n)[-1] if isinstance(n, str) else n for n in row.display_name_alternatives]
        print(f'{authors.shape = }\n{authors.head()}')

        for row in sample.itertuples():
            for row1 in authors.itertuples():
                if row.fullname in row1.display_name_alternatives:
                    sample.at[row.Index, 'author_id'] = row1.author_id
                    sample.at[row.Index, 'orcid'] = row1.orcid
                    break
        print(f'{sample.shape = }\n{sample.head(24)}')
        print(f'{sample[sample.author_id.isna()].shape = }\n{sample[sample.author_id.isna()].head(24)}')
        self.db.sql("CREATE OR REPLACE TABLE memory.matched AS SELECT * FROM sample")
        return

    def load_sample(self):
        print('compare sample')
        sql = """
            CREATE OR REPLACE TABLE econ.sample_matched AS
                SELECT Research_Profile,
                        m.orcid,
                        m.author_id,
                        m.first,
                        m.last,
                        m.fullname,
                        ACR,
                        PUB,
                        CIT,
                        HCP,
                        suma,          
                        coc,      
                        score, 
                        "Group",
                        works_count_endogenous, 
                        citations_endogenous,
                        hca_endogenous,	                    	
                        works_count_total,
                        cited_by_count,
                        hca_total,
                        "2yr_mean_citedness",
                        h_index,
                        citations_total_ AS citations_total_oa,	
                    FROM memory.matched m 
                    LEFT JOIN citation_summary s
                        ON s.author_id = m.author_id
                    ORDER BY hca_endogenous DESC, citations_endogenous DESC
            """
        self.db.sql(sql)
        sample_align = self.db.sql("SELECT * FROM econ.sample_matched").df()
        print(f'{sample_align.shape = }\n{sample_align.head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample_align.to_excel(writer, index=False, sheet_name='full_match')
            df = sample_align.groupby('Research_Profile').first().reset_index()
            df.to_excel(writer, index=False, sheet_name='filtered')
            print(f'{df.shape = }\n{df.head()}')
            df[df.author_id.isna()].to_excel(writer, index=False, sheet_name='filtered_unmatched')
            print(f'{df[df.author_id.isna()].shape = }\n{df[df.author_id.isna()].head()}')
        return

In [129]:
def main():

    jetl = ArticlesETL()
    # jetl.extract_journals()
    # jetl.extract_works_by_journal()
    print(jetl.db.sql("DESCRIBE TABLE project.raw").df())
    jetl.db.sql("SELECT * FROM project.raw").show()
    sql = """SELECT count(DISTINCT "primary_location.source"['id']) FROM project.raw"""
    jetl.db.sql(sql).show()
    jetl.db.close()

    # ea = ExtractAuthors()
    # # ea.build_author_list()
    # # ea.extract_authors()
    # ea.db.close()

    # mds = MatchDomingoSample()        # cols = [c for c in works.columns if ('abstract_inverted_index' not in c) or (c in keep_columns)]
    # mds.db.close()

In [130]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬────────────────────┬──────────────────────┬─────────────────────────────────────────┬───────────┐
│ database │ schema  │        name        │     column_names     │              column_types               │ temporary │
│ varchar  │ varchar │      varchar       │      varchar[]       │                varchar[]                │  boolean  │
├──────────┼─────────┼────────────────────┼──────────────────────┼─────────────────────────────────────────┼───────────┤
│ project  │ main    │ raw                │ [id, doi, title, p…  │ [VARCHAR, VARCHAR, VARCHAR, BIGINT, V…  │ false     │
│ project  │ main    │ sources_oa_incites │ [column00, id, iss…  │ [BIGINT, VARCHAR, VARCHAR, VARCHAR, V…  │ false     │
│ project  │ main    │ works_transform    │ [work_id, publicat…  │ [VARCHAR, BIGINT, VARCHAR, VARCHAR, B…  │ false     │
└──────────┴─────────┴────────────────────┴──────────────────────┴─────────────────────────────────────────┴───────────┘

self.cache.check() = []
self.ca